# Accuracy, Precision, and Recall

**Accuracy** measures overall correctness (correct predictions out of total predictions): `(TP + TN) / (TP + TN + FP + FN)`

-> Example: Of all tested people, how many states of infections (both healthy *and* infected) did the Corona test get right?

**Precision** measures how many predictions of a class were actually of that class:
`TP / (TP + FP)`

-> Example: Of all people *tested positively*, how many people were *actually* infected?

**Recall** measures the percentage of instances of a class that were identified correctly: `TP / (TP + FN)`

-> Example: Of all *actually* infected people, how many were *tested* positively?


## A Simple Example on Fraud Detection

We build a simple toy dataset with arbitrary transaction values and fraud labels (1 for fraud).

In [1]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.metrics import accuracy_score, precision_score, recall_score

# Assume this is for one bank account - usually transactions are below 1000€ (label 0, means "no fraud")
credit_card_transaction_df = pd.DataFrame({
  'Transactions [€]': np.random.randint(1, 1000, 990).tolist() + np.random.randint(50_000, 100_000, 9).tolist() + [500_000],
  'Labels': [0]*990 + [1]*10
})

label_dict = {0: 'no fraud', 1: 'fraud'}

credit_card_transaction_df.head()

,Transactions [€],Labels
0,205,0
1,136,0
2,584,0
3,868,0
4,105,0


Now we want to build a classifier that predicts the label of transactions.<br>
Let's optimize for accuracy.<br>

In [2]:
# lambda function is a simple classifier that labels no fraud as long as transaction amount is below 100,000€
clf_accuracy = lambda x: 0 if x < 100_000 else 1

y_pred_acc = credit_card_transaction_df['Transactions [€]'].apply(clf_accuracy)
accuracy_acc = accuracy_score(credit_card_transaction_df['Labels'], y_pred_acc)
print(f'Accuracy of the classifier: {accuracy_acc*100:.1f}%')

Accuracy of the classifier: 99.1%


Great! Above 99% accuracy. I guess we are done here.<br>
But to be sure, let's do a quick sanity check on a 'fraud' sample.

In [3]:
fraud_sample = credit_card_transaction_df[credit_card_transaction_df['Transactions [€]'] > 10_000]
fraud_pred = fraud_sample['Transactions [€]'].apply(clf_accuracy)
print(f'Predictions for fraud sample:\n{label_dict[fraud_pred.iloc[0]]}')

Predictions for fraud sample:
no fraud


That is annoying...<br>
Let's see if other performance metrics give a better indicator

In [4]:
# Apply the classifier to the transaction data
precision = precision_score(credit_card_transaction_df['Labels'], y_pred_acc)
recall = recall_score(credit_card_transaction_df['Labels'], y_pred_acc)

print(f'Precision of the classifier: {precision*100:.1f}%')
print(f'Recall of the classifier: {recall*100:.1f}%')

Precision of the classifier: 100.0%
Recall of the classifier: 10.0%


Seems like precision doesn't fix our problem here.<br>
But what about recall?
The score seems to indicate our model is pretty terrible.

---
Let's optimize for recall!

In [5]:
# lambda function is a simple classifier that labels fraud if transaction amount is greater than 1€
clf_rec = lambda x: 1 if x > 1 else 0

# Apply the classifier to the transaction data
y_pred_rec = credit_card_transaction_df['Transactions [€]'].apply(clf_rec)
recall_2 = recall_score(credit_card_transaction_df['Labels'], y_pred_rec)

print(f'Recall of the classifier after optimization: {recall_2*100:.1f}%')

Recall of the classifier after optimization: 100.0%


Amazing! We reached a recall score of 100% and are able to identify all fraudulent transcations.<br>
Let's check the other metrics to be sure.

In [6]:
accuracy_acc = accuracy_score(credit_card_transaction_df['Labels'], y_pred_rec)
print(f'Accuracy of the classifier: {accuracy_acc*100:.1f}%')

precision = precision_score(credit_card_transaction_df['Labels'], y_pred_rec)
print(f'Precision of the classifier: {precision*100:.1f}%')

Accuracy of the classifier: 1.3%
Precision of the classifier: 1.0%


Oh well... Seems like optimizing for recall alone also doesn't do the trick.<br>
Let's have a closer look.

In [7]:
from sklearn.metrics import classification_report

# For the first model
clf_report_acc = classification_report(credit_card_transaction_df['Labels'], y_pred_acc, target_names=['no fraud', 'fraud'])
print(clf_report_acc)

# For the second model
clf_report_rec = classification_report(credit_card_transaction_df['Labels'], y_pred_rec, target_names=['no fraud', 'fraud'])
print(clf_report_rec)

              precision    recall  f1-score   support

    no fraud       0.99      1.00      1.00       990
       fraud       1.00      0.10      0.18        10

    accuracy                           0.99      1000
   macro avg       1.00      0.55      0.59      1000
weighted avg       0.99      0.99      0.99      1000

              precision    recall  f1-score   support

    no fraud       1.00      0.00      0.01       990
       fraud       0.01      1.00      0.02        10

    accuracy                           0.01      1000
   macro avg       0.51      0.50      0.01      1000
weighted avg       0.99      0.01      0.01      1000



In the classification report, we can see a distinction between the different classes (0 and 1).<br>

The first model optimized for accuracy doesn't predict the minority class 'fraud' very well.<br>

The second model optimized for recall doesn't predict the majority class 'no fraud' very well.<br>

In the end you have to be conscious about your goals. Do you want to make sure to capture ALL labels of a certain class? Or do you need an overall good performance?<br>
Either way its good practice to check more than one metric.